# Avaliação final do detector
Execute em ordem para reproduzir a tabela e conferir a contagem do teste a partir dos resultados arquivados. Usa Python 3 e sua biblioteca padrão. O treino e a inferência originais estão nos scripts do repositório. Para executar fora do clone, a célula de leitura consulta os CSVs do commit avaliado no GitHub.

In [1]:
import csv
import io
import sys
from pathlib import Path
from urllib.request import urlopen

print(sys.version.split()[0])
folder = Path("analysis/final_eval_315")
if not folder.is_dir() and (Path("..") / folder).is_dir():
    folder = Path("..") / folder

def read_rows(filename):
    path = folder / filename
    if path.is_file():
        content = path.read_text(encoding="utf-8")
    else:
        base = "https://raw.githubusercontent.com/renatocorreia-rmcm/allnomalous/757b587/analysis/final_eval_315/"
        with urlopen(base + filename, timeout=30) as response:
            content = response.read().decode("utf-8")
    return list(csv.DictReader(io.StringIO(content)))

3.13.1


## Comparação por categoria
Valores percentuais. Para cada métrica, as colunas mostram baseline e completo, nessa ordem.

In [2]:
metrics = read_rows("final_metrics.csv")
by_model = {(r["model"], r["category"]): r for r in metrics}
print(f"{'Categoria':12}  {'AUROC imagem':>15}  {'AUROC pixel':>15}  {'AUPRO':>15}")
for row in metrics:
    if row["model"] == "baseline":
        category = row["category"]
        pairs = []
        for key in ("image_auroc", "pixel_auroc", "aupro"):
            values = [100 * float(by_model[model, category][key]) for model in ("baseline", "complete")]
            pairs.append(" / ".join(f"{value:.2f}" for value in values))
        print(f"{category:12}  " + "  ".join(pairs))

Categoria        AUROC imagem      AUROC pixel            AUPRO
bottle        96.35 / 96.51  91.09 / 91.02  84.10 / 84.10
cable         71.68 / 71.01  73.33 / 73.12  65.75 / 65.88
capsule       64.62 / 65.10  93.04 / 93.12  84.32 / 84.56
carpet        89.73 / 89.17  94.88 / 94.50  86.28 / 85.81
grid          85.05 / 85.71  94.73 / 95.20  81.37 / 82.89
hazelnut      97.86 / 98.00  95.61 / 95.67  90.28 / 90.44
leather       94.53 / 94.16  97.51 / 97.51  93.95 / 94.02
metal_nut     57.33 / 56.94  81.50 / 81.34  63.52 / 63.52
pill          66.18 / 66.83  86.07 / 86.19  86.22 / 86.35
screw         51.22 / 51.71  95.46 / 95.54  84.24 / 84.48
tile          94.08 / 93.80  87.19 / 86.82  68.45 / 67.95
toothbrush    80.00 / 80.56  92.35 / 92.59  72.28 / 72.87
transistor    56.29 / 56.17  65.99 / 66.06  43.38 / 43.77
wood          97.63 / 97.54  93.05 / 93.00  91.96 / 91.93
zipper        73.45 / 73.69  94.38 / 94.50  82.25 / 82.67
macro         78.40 / 78.46  89.08 / 89.08  78.56 / 78.75


## Conferência do teste pareado

In [3]:
scores = read_rows("escores_ambos.csv")
def labels(model):
    rows = [r for r in scores if r["modelo"] == model]
    result = {r["imagem"]: int(r["rotulo"]) for r in rows}
    assert len(rows) == len(result) == 1725
    return result

baseline = labels("baseline")
assert baseline == labels("complete")
print(f"{sum(baseline.values())} defeituosas e {len(baseline)-sum(baseline.values())} normais.")

1258 defeituosas e 467 normais.
